# Inflation-linked bonds

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`. **`inflation_linked_bond`** needs a **real discount** curve plus a typed **`InflationIndex`** in the `MarketContext`.


## Concept

- **Real coupon** plus **index ratio** from CPI fixes cashflows.
- **Real yield** and **breakeven inflation** are interpreted relative to nominal curves (this notebook prints registry metrics when available).
- The typed inflation index uses the Rust default of no lag because the bond applies its own lag (avoids double-lagging).


In [ ]:
import json
from datetime import date

AS_OF = date(2025, 1, 15)

from _shared import print_metrics
from finstack_quant.valuations import ValuationResult
from finstack_quant.core.market_data import DiscountCurve, InflationCurve, InflationIndex, InflationObservation, InflationExpectedRelease, MarketContext, VolSurface

print("Imports OK (inflation-linked).")

## Build the bond

In [ ]:
from datetime import date
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

from decimal import Decimal
from finstack_quant.core.money import Money
from finstack_quant.valuations.instruments import InflationLinkedBond, PriceExtras, price_instrument

ilb = (
    InflationLinkedBond.builder()
    .id("TIPS-DEMO")
    .notional(Money(1_000_000, "USD"))
    .issue_date("2024-01-15")
    .maturity("2034-01-15")
    .real_coupon(Decimal("0.025"))
    .frequency("6M")
    .day_count("30_360")
    .business_day_convention("following")
    .calendar_id("weekends_only")
    .stub("none")
    .base_date("2024-01-15")
    .base_cpi(277.0676975464149)
    .indexation_method("tips")
    .inflation_index_id("US-CPI")
    .lag("3M")
    .deflation_protection("maturity_only")
    .floor_vol_surface_id("CPI-ZERO-VOL")
    .discount_curve_id("USD-OIS")
    .build()
)
price_extras = PriceExtras(
    instrument_pricing_overrides={"market_quotes": {"quoted_clean_price_pct": 100.0}}
)
assert ilb.real_coupon == Decimal("0.025")


## MarketContext


In [ ]:
tips = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85), (10.0, 0.70)],
    day_count="act_365f",
    interp="log_linear",
    extrapolation="extend",
)
obs = []
y, m, v = 2019, 10, 250.0
end = date(2035, 1, 1)
while date(y, m, 1) <= end:
    obs.append((date(y, m, 1), v))
    m += 1
    if m > 12:
        m, y = 1, y + 1
    v *= 1.002
def release_date(month):
    return date(month.year + (month.month == 12), month.month % 12 + 1, 15)
history = [InflationObservation(d, val, release_date(d)) for d, val in obs if release_date(d) <= AS_OF]
releases = [InflationExpectedRelease(d, release_date(d), AS_OF) for d, _ in obs]
inflation_index = InflationIndex("US-CPI", history, "USD", AS_OF).with_expected_releases(releases)
inflation_curve = InflationCurve.from_dates("US-CPI", date(2019, 10, 1), obs, day_count="act_365f", interp="log_linear", extrapolation="extend")
zero_floor_vol = VolSurface("CPI-ZERO-VOL", [0.01, 100.0], [0.5, 1.5], [[0.0, 0.0], [0.0, 0.0]], secondary_axis="strike", quote_type="black_lognormal", interpolation_mode="vol")
mc = MarketContext().insert(tips).insert(inflation_curve).insert(zero_floor_vol)
mc.insert_inflation_index(inflation_index)
print("inflation index observations:", len(mc.get_inflation_index("US-CPI")))


## Pricing


In [ ]:
out = price_instrument(ilb, mc, AS_OF_STR, extras=price_extras, model="discounting")
print(out)


## Metrics


In [ ]:
metrics = ["breakeven_inflation", "index_ratio"]
out = price_instrument(
    ilb,
    # The quoted clean price is an explicit pricing input.
    mc,
    AS_OF_STR,
    model="discounting",
    metrics=metrics,
    extras=price_extras,
)
print_metrics(out, metrics)
print("inflation metrics pass complete")


## Price a zero-coupon inflation swap

The swap pays a fixed amount and receives the cumulative move in the same CPI index used for the bond. The builder preserves the example trade’s observation lag and interpolation.

In [ ]:
from math import isfinite
from finstack_quant.valuations.instruments import InflationSwap

swap = (
    InflationSwap.example()
    .to_builder()
    .id("US-CPI-PAY-FIXED")
    .fixed_rate(Decimal("0.025"))
    .build()
)
swap_result = price_instrument(swap, mc, AS_OF)
assert isfinite(swap_result.price)
print(f"CPI swap PV: USD {swap_result.price:,.2f}")


## Takeaways

- **Inflation indices** ride alongside curves in the v2 market snapshot.
- **Breakeven** and **index ratio** metrics connect linker pricing to macro intuition.
- Match **lag policy** between bond spec and index payload to avoid double-lag validation errors.
